In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq  # <--- CHANGED THIS
)

# --- Configuration ---
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
NEW_MODEL_NAME = "Qwen2.5-0.5B-Abstract-to-Title"
DATASET_FILE = "d.csv"
MAX_SEQ_LENGTH = 1024

# --- 1. Load Tokenizer & Model ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# FIX: Qwen often lacks a pad token. We MUST set it for batch training to work.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    # We also explicitly tell the model about this change
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# --- 2. Load & Format Dataset ---
dataset = load_dataset("csv", data_files=DATASET_FILE, split="train")

def format_prompts(example):
    abstract = example['abstract']
    title = example['title']
    
    messages = [
        {"role": "system", "content": "Generate a concise and accurate title for the following research paper abstract."},
        {"role": "user", "content": abstract},
        {"role": "assistant", "content": title}
    ]
    
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=False
    )
    
    # We keep padding=False here so we don't waste memory. 
    # The DataCollator will handle padding dynamically per batch.
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False, 
        add_special_tokens=True
    )
    
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Formatting dataset...")
tokenized_dataset = dataset.map(format_prompts, remove_columns=dataset.column_names)

# --- 3. Data Collator (THE FIX) ---
# DataCollatorForSeq2Seq handles dynamic padding for both inputs and labels perfectly.
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100  # PyTorch ignores -100 in loss calculation
)

# --- 4. Training Arguments ---
training_args = TrainingArguments(
    output_dir="./title_gen_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Try 2 or 1 if you still get memory errors
    gradient_accumulation_steps=8,  # Accumulate more steps to compensate for small batch size
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,                      # Use fp16=True if on older GPU (T4/V100)
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False     # Important prevents Trainer from dropping our 'labels'
)

# --- 5. Initialize Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=collator, # Use the fixed collator
)

# --- 6. Train ---
print("Starting training...")
trainer.train()

print(f"Saving model to {NEW_MODEL_NAME}...")
trainer.save_model(NEW_MODEL_NAME)
tokenizer.save_pretrained(NEW_MODEL_NAME)
print("Done!")

/home/matheusc/Repos/ufmt.topicos.ia.trabalho.final/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
`torch_dtype` is deprecated! Use `dtype` instead!
The model is already on multiple devices. Skipping the move to device specified in `args`.


Formatting dataset...
Starting training...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
5,3.173800
10,3.057400
15,3.037600
20,2.824400
25,2.722400
30,2.574100
35,2.441000
40,2.311700
45,2.204600
50,2.234900


Saving model to Qwen2.5-0.5B-Abstract-to-Title...
Done!


In [1]:
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# --- Configuration ---
# 1. We load the TOKENIZER from the original hub (Bypasses the "dict" error)
ORIGINAL_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# 2. We load the TRAINED MODEL from your local folder
LOCAL_MODEL_PATH = "Qwen2.5-0.5B-Abstract-to-Title"

INPUT_CSV = "datasets/test_data.csv"
OUTPUT_CSV = "results_predictions.csv"

# --- 1. Load Tokenizer & Model ---
print(f"Loading tokenizer from {ORIGINAL_MODEL_ID}...")
# Trust remote code is needed for Qwen
tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_MODEL_ID, trust_remote_code=True)

# Ensure pad token is set correctly
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Loading model from {LOCAL_MODEL_PATH}...")
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# --- 2. Load Data ---
try:
    df = pd.read_csv(INPUT_CSV)
    print(f"Loaded {len(df)} rows from {INPUT_CSV}")
except FileNotFoundError:
    print(f"Error: Could not find {INPUT_CSV}. Make sure the file exists.")
    exit()

predictions = []

# --- 3. Inference Loop ---
print("Starting inference...")

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    # Robust check: ensure abstract is a string
    abstract = str(row['abstract'])
    
    # Prompt Format
    messages = [
        {"role": "system", "content": "Generate a concise and accurate title for the following research paper abstract."},
        {"role": "user", "content": abstract}
    ]

    # Apply Template
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Tokenize
    # max_length=32768 allows the full Qwen context window (no practical limit on reading)
    model_inputs = tokenizer([text], return_tensors="pt", max_length=32768, truncation=True).to(model.device)

    # Generate
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512,  # Allow long titles if necessary
            do_sample=False,     # Greedy decoding for the most "likely" title (better for accuracy)
            repetition_penalty=1.1 # Helps prevent looping on long abstracts
        )

    # Decode
    # Remove the input tokens from the output
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    predicted_title = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    predictions.append(predicted_title.strip())

# --- 4. Save Results ---
df['predicted_title'] = predictions
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nDone! Results saved to {OUTPUT_CSV}")

/home/matheusc/Repos/ufmt.topicos.ia.trabalho.final/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer from Qwen/Qwen2.5-0.5B-Instruct...
Loading model from Qwen2.5-0.5B-Abstract-to-Title...


Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loaded 1000 rows from datasets/test_data.csv
Starting inference...


  0%|                                                                                                                        | 0/1000 [00:00<?, ?it/s]/home/matheusc/Repos/ufmt.topicos.ia.trabalho.final/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/matheusc/Repos/ufmt.topicos.ia.trabalho.final/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/matheusc/Repos/ufmt.topicos.ia.trabalho.final/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do


Done! Results saved to results_predictions.csv
